# Thermoelectric residual-learning pipeline — full, run-it-yourself version

This notebook runs the **entire** pipeline from raw data to the current
end-point (XGBoost vs. a physically-restricted symbolic regression), so you
can verify every number yourself in your own Colab session rather than
trusting a report. Nothing here is pre-filled with output — every cell is
meant to be run top to bottom.

**Files you need to upload when Cell 2 asks for them** (all of these you
already have, since they're the same files used throughout this project):
- `raw_starrydata2_samples.csv.gz`
- `raw_starrydata2_curves.csv.gz`
- `tier2_input_halfheusler_samples.csv`
- the merged workbook that contains the `ricci_dft_clean_halfheusler_1` and
  `ricci_dft_clean_skutterudite_1` sheets (your `merged-output.xlsx`)

**What this notebook does, step by step:**
1. Install packages, load the raw data
2. Clean the data (two bug fixes, both explained inline)
3. Fit the physics baseline (SPB model) per family and carrier type
4. Compute the residual = measured − baseline
5. Cross-check the fitted effective mass against independent DFT values
6. Test whether the residual relates to fabrication process (raw categories, then a physically-grounded 3-group version)
7. XGBoost: how much of the residual can be predicted from family / carrier type / temperature / carrier concentration, with and without the fabrication-process label
8. Symbolic regression, restricted to real physical quantities only (temperature, carrier concentration — never a category label), as the interpretable counterpart to XGBoost
9. How to read and compare the results


## Step 0: install packages

In [1]:
!pip install -q xgboost gplearn openpyxl
import numpy as np
import pandas as pd
from scipy.integrate import quad
from scipy.interpolate import interp1d
from scipy.optimize import minimize_scalar
from scipy import stats
import warnings
warnings.filterwarnings("ignore")
print("Packages ready.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 55.5 MB/s eta 0:00:00
Packages ready.


## Upload your data files

In [5]:
try:
    from google.colab import files
    print("Upload raw_starrydata2_samples.csv.gz, raw_starrydata2_curves.csv.gz, "
          "tier2_input_halfheusler_samples.csv, and your merged-output.xlsx (the one with the ricci_dft_clean sheets).")
    uploaded = files.upload()
    for fn in uploaded:
        print("Uploaded:", fn)
except ImportError:
    print("Not running in Colab -- assuming the files are already in the working directory.")

SAMPLES_PATH = "raw_starrydata2_samples.csv.gz"
CURVES_PATH = "raw_starrydata2_curves.csv.gz"
TIER2_INPUT_PATH = "tier2_input_halfheusler_samples.csv"
# If your merged workbook has a different filename, change this:
MERGED_XLSX_PATH = "merged-output.xlsx"


Upload raw_starrydata2_samples.csv.gz, raw_starrydata2_curves.csv.gz, tier2_input_halfheusler_samples.csv, and your merged-output.xlsx (the one with the ricci_dft_clean sheets).


Saving merged-output.xlsx to merged-output.xlsx
Saving raw_starrydata2_curves.csv.gz to raw_starrydata2_curves.csv.gz
Saving raw_starrydata2_samples.csv.gz to raw_starrydata2_samples.csv (1).gz
Saving tier2_input_halfheusler_samples.csv to tier2_input_halfheusler_samples.csv
Uploaded: merged-output.xlsx
Uploaded: raw_starrydata2_curves.csv.gz
Uploaded: raw_starrydata2_samples.csv (1).gz
Uploaded: tier2_input_halfheusler_samples.csv


## Steps 1-2: load and clean

**Fix 1** — the original pipeline's carrier-concentration sanity filter was
`1e14 < n < 1e23` (m⁻³). Auditing the raw dataset shows real doped-
thermoelectric carrier concentrations actually sit at roughly `1e23-1e28`
m⁻³ — the old filter was keeping the wrong end of the distribution and
throwing away most of the good data. Corrected here to `1e18 < n < 1e27`.

**Fix 2** — the effective mass (m*) fit was pooling n-type and p-type
carriers under one shared value per family, which is not physically
defensible (they have genuinely different band-edge masses) and was causing
the optimizer to saturate at its search bounds. Fixed by fitting m*
separately per (family, carrier-sign) group.

In [ ]:
KB = 1.380649e-23; E = 1.602176634e-19; H = 6.62607015e-34; ME = 9.1093837015e-31

def fermi_dirac(j, eta):
    f = lambda x: (x ** j) / (1.0 + np.exp(np.clip(x - eta, -700, 700)))
    val, _ = quad(f, 0, np.inf, limit=200)
    return val

ETA_GRID = np.linspace(-15, 20, 400)
F0 = np.array([fermi_dirac(0.0, e) for e in ETA_GRID])
F1 = np.array([fermi_dirac(1.0, e) for e in ETA_GRID])
FHALF = np.array([fermi_dirac(0.5, e) for e in ETA_GRID])
S_GRID = (KB / E) * (2.0 * F1 / F0 - ETA_GRID)

eta_of_lnFhalf = interp1d(np.log(FHALF), ETA_GRID, bounds_error=False,
                           fill_value=(ETA_GRID[0], ETA_GRID[-1]))
S_of_eta = interp1d(ETA_GRID, S_GRID, bounds_error=False,
                     fill_value=(S_GRID[0], S_GRID[-1]))
eta_of_S = interp1d(S_GRID[::-1], ETA_GRID[::-1], bounds_error=False,
                     fill_value=(ETA_GRID[-1], ETA_GRID[0]))

def spb_seebeck(n_m3, T, mstar_ratio):
    mstar = mstar_ratio * ME
    Nc = 4.0 * np.pi * (2.0 * mstar * KB * T / H ** 2) ** 1.5
    n_reduced = np.clip(n_m3 / Nc, 1e-6, None)
    eta = eta_of_lnFhalf(np.log(n_reduced))
    return S_of_eta(eta)

def fit_mstar(T, n_m3, S_abs):
    def loss(mstar_ratio):
        pred = spb_seebeck(n_m3, T, mstar_ratio)
        return np.sum((pred - S_abs) ** 2)
    res = minimize_scalar(loss, bounds=(0.02, 25.0), method="bounded")
    return res.x

print("Physics helper functions defined.")


In [6]:
import ast, json, re

def parse_info(info):
    if not isinstance(info, str) or not info.strip():
        return {}
    try:
        return json.loads(info)
    except Exception:
        return {}

def parse_arr(s):
    try:
        return np.array(ast.literal_eval(s), dtype=float)
    except Exception:
        return None

print("Loading samples table...")
samples = pd.read_csv(SAMPLES_PATH)
samples["info"] = samples["sample_info"].apply(parse_info)
samples["family_tag"] = samples["info"].apply(lambda d: (d.get("MaterialFamily") or {}).get("category") or None)
samples["fab_cat"] = samples["info"].apply(lambda d: (d.get("FabricationProcess") or {}).get("category") or None)
samples["fab_comment"] = samples["info"].apply(lambda d: (d.get("FabricationProcess") or {}).get("comment") or None)
samples["density"] = samples["info"].apply(lambda d: (d.get("RelativeDensity") or {}).get("extracted") or None)
samples["grainsize"] = samples["info"].apply(lambda d: (d.get("GrainSize") or {}).get("extracted") or None)
samples = samples.drop(columns=["info"])

print("Loading curves table (this is the big one)...")
curves = pd.read_csv(CURVES_PATH, usecols=["sample_id", "prop_x", "prop_y", "x", "y", "composition"])

seebeck_curves = curves[(curves.prop_x == "Temperature") & (curves.prop_y == "Seebeck coefficient")]
carrier_curves = curves[(curves.prop_x == "Temperature") & (curves.prop_y == "Carrier concentration")]
hall_curves = curves[(curves.prop_x == "Temperature") & (curves.prop_y == "Hall coefficient")]

print(f"Seebeck(T) curves: {len(seebeck_curves)}   Carrier-conc(T) curves: {len(carrier_curves)}   "
      f"Hall(T) curves (as n proxy): {len(hall_curves)}")

carrier_by_sample = {}
def add_points(sid, tx, ny):
    if sid not in carrier_by_sample:
        carrier_by_sample[sid] = ([], [])
    carrier_by_sample[sid][0].append(tx)
    carrier_by_sample[sid][1].append(ny)

for _, row in carrier_curves.iterrows():
    tx, ny = parse_arr(row["x"]), parse_arr(row["y"])
    if tx is None or ny is None or len(tx) < 2:
        continue
    add_points(row["sample_id"], tx, ny)

for _, row in hall_curves.iterrows():
    tx, rh = parse_arr(row["x"]), parse_arr(row["y"])
    if tx is None or rh is None or len(tx) < 2:
        continue
    rh = np.abs(rh)
    rh = np.where(rh > 0, rh, np.nan)
    n_from_hall = 1.0 / (rh * E)
    add_points(row["sample_id"], tx, n_from_hall)

carrier_by_sample = {sid: (np.concatenate(ts), np.concatenate(ns)) for sid, (ts, ns) in carrier_by_sample.items()}
print(f"Samples with a usable carrier-concentration(T) source: {len(carrier_by_sample)}")


Loading samples table...
Loading curves table (this is the big one)...
Seebeck(T) curves: 36112   Carrier-conc(T) curves: 1921   Hall(T) curves (as n proxy): 587
Samples with a usable carrier-concentration(T) source: 2110


In [7]:
FAMILY_RULES = [
    ("Half-Heusler", re.compile(r"(Ti|Zr|Hf|V|Nb|Ta|Sc|Y|La|Ce)\w*.*(Ni|Co|Fe|Pd|Pt|Rh)\w*.*(Sn|Sb|Bi|As|Ge)")),
    ("Skutterudite", re.compile(r"(Co|Rh|Ir)\d*\.?\d*Sb\d*\.?\d{2,}")),
    ("Bi2Te3-based", re.compile(r"Bi\d*\.?\d*.*Te\d")),
    ("Sb2Te3/Ge-Sb-Te", re.compile(r"Sb\d*\.?\d*Te\d")),
    ("PbTe-based", re.compile(r"Pb\d*\.?\d*Te\d")),
    ("SnSe-based", re.compile(r"Sn\d*\.?\d*Se\d")),
    ("SnS-based", re.compile(r"Sn\d*\.?\d*S\d(?!e)")),
    ("SnTe-based", re.compile(r"Sn\d*\.?\d*Te\d")),
    ("GeTe/GeSe", re.compile(r"Ge\d*\.?\d*(Te|Se)\d")),
    ("SiGe alloy", re.compile(r"^(Si\d*\.?\d*Ge\d*\.?\d*|Ge\d*\.?\d*Si\d*\.?\d*)$")),
    ("Silicide (CrSi2/FeSi2 etc.)", re.compile(r"(Cr|Fe|Mn|Mg)\d*\.?\d*Si\d")),
    ("Mg2Si/Sn", re.compile(r"Mg2(Si|Sn)")),
    ("Zintl (Ca/Sr/Eu/Yb-Zn/Al-Sb)", re.compile(r"(Ca|Sr|Eu|Yb)\d*\.?\d*.*(Zn|Al|Cd)\d*\.?\d*Sb")),
    ("Cu-chalcogenide (Cu2Se/CuInTe2 etc.)", re.compile(r"Cu\d*\.?\d*(Se|S|Te|In)")),
    ("In-Te/In-Se chalcogenide", re.compile(r"^In\d*\.?\d*(Te|Se)")),
    ("Ag-chalcogenide", re.compile(r"^Ag\d*\.?\d*(Ga|S|Se|Te)")),
    ("Oxide/Cobaltite", re.compile(r"(Co|Zn)\w*O\d|^Zn?O$|CrO|Pb\d*\.?\d*CrO")),
]

def classify_family(tag, composition):
    if isinstance(tag, str) and tag.strip():
        return tag
    comp = str(composition)
    for name, pat in FAMILY_RULES:
        if pat.search(comp):
            return name
    return None

samples["family_assigned"] = samples.apply(lambda r: classify_family(r["family_tag"], r["composition"]), axis=1)

def extract_matched_points(seebeck_row_group, min_T=150, max_T=1200):
    sid = seebeck_row_group.name
    if sid not in carrier_by_sample:
        return None
    Tn, nn = carrier_by_sample[sid]
    order = np.argsort(Tn)
    Tn, nn = Tn[order], nn[order]
    Tn_u, idx = np.unique(Tn, return_index=True)
    nn_u = nn[idx]
    if len(Tn_u) < 2:
        return None
    n_interp = interp1d(Tn_u, nn_u, bounds_error=False, fill_value=np.nan)
    rows = []
    for _, row in seebeck_row_group.iterrows():
        Ts, Ss = parse_arr(row["x"]), parse_arr(row["y"])
        if Ts is None or Ss is None:
            continue
        for T, S in zip(Ts, Ss):
            if not (min_T <= T <= max_T):
                continue
            if T < Tn_u.min() or T > Tn_u.max():
                continue
            n_val = float(n_interp(T))
            if not np.isfinite(n_val) or n_val <= 0:
                continue
            rows.append((sid, T, S, n_val))
    return rows

print("Matching Seebeck and carrier-concentration curves per sample...")
all_rows = []
for sid, grp in seebeck_curves.groupby("sample_id"):
    grp.name = sid
    r = extract_matched_points(grp)
    if r:
        all_rows.extend(r)

matched = pd.DataFrame(all_rows, columns=["sample_id", "T", "S_measured", "n_measured"])
print(f"Matched (T, S, n) points before quality filtering: {len(matched)}")

# --- FIX 1 here ---
matched = matched[(matched["S_measured"].abs() < 0.002) &
                   (matched["n_measured"] > 1e18) & (matched["n_measured"] < 1e27)]
print(f"Matched points after CORRECTED filtering (n in [1e18,1e27] m^-3): {len(matched)}")

matched = matched.merge(
    samples[["sample_id", "composition", "family_tag", "family_assigned", "fab_cat", "fab_comment", "density", "grainsize"]],
    on="sample_id", how="left")


Matching Seebeck and carrier-concentration curves per sample...
Matched (T, S, n) points before quality filtering: 18968
Matched points after CORRECTED filtering (n in [1e18,1e27] m^-3): 16052


## Steps 3-4: fit the SPB baseline and compute the residual (Tier 1: all families pooled)

In [8]:
tier1 = matched.dropna(subset=["family_assigned"]).copy()
tier1 = tier1.rename(columns={"family_assigned": "family"})
family_counts = tier1.groupby("family")["sample_id"].nunique().sort_values(ascending=False)
KEEP_FAMILIES = family_counts[family_counts >= 3].index.tolist()
tier1 = tier1[tier1["family"].isin(KEEP_FAMILIES)].copy()

print(f"TIER 1: {tier1['sample_id'].nunique()} samples, {tier1['family'].nunique()} families, {len(tier1)} (T,S,n) points")

tier1["n_cm3"] = tier1["n_measured"] / 1e6
tier1["log10_n"] = np.log10(tier1["n_cm3"])

# --- FIX 2 here: fit m* per (family, carrier-sign), not pooled ---
tier1_out = []
for (fam, sgn), grp in tier1.groupby(["family", np.sign(tier1["S_measured"])]):
    T = grp["T"].values
    n_m3 = grp["n_measured"].values
    S_abs = grp["S_measured"].abs().values
    if len(grp) >= 3:
        mstar = fit_mstar(T, n_m3, S_abs)
    else:
        fam_grp = tier1[tier1["family"] == fam]
        mstar = fit_mstar(fam_grp["T"].values, fam_grp["n_measured"].values,
                           fam_grp["S_measured"].abs().values) if len(fam_grp) >= 3 else 1.0
    pred = spb_seebeck(n_m3, T, mstar)
    signed_pred = np.sign(grp["S_measured"].values) * pred
    g = grp.copy()
    g["carrier_type"] = np.where(sgn > 0, "p-type", "n-type")
    g["mstar_over_me"] = mstar
    g["S_baseline_V_per_K"] = signed_pred
    g["S_residual_V_per_K"] = g["S_measured"] - g["S_baseline_V_per_K"]
    tier1_out.append(g)
tier1_final = pd.concat(tier1_out, ignore_index=True)

resid_uVK = tier1_final["S_residual_V_per_K"] * 1e6
print(f"TIER 1 SPB baseline fit: RMSE={np.sqrt(np.mean(resid_uVK**2)):.1f} uV/K, "
      f"MAE={np.mean(np.abs(resid_uVK)):.1f} uV/K "
      f"(measured |S| median = {tier1_final['S_measured'].abs().median()*1e6:.1f} uV/K)")
tier1_final.to_csv("tier1_pooled_baseline_residuals.csv", index=False)


TIER 1: 967 samples, 37 families, 12358 (T,S,n) points
TIER 1 SPB baseline fit: RMSE=138.8 uV/K, MAE=71.7 uV/K (measured |S| median = 158.4 uV/K)


### Tier 2: half-Heusler only (needed for the DFT cross-check in Step 5)

In [9]:
hh_all = pd.read_csv(TIER2_INPUT_PATH)
hh_ids = set(hh_all["sample_id"])
tier2 = matched[matched["sample_id"].isin(hh_ids)].copy()
tier2["n_cm3"] = tier2["n_measured"] / 1e6
tier2["log10_n"] = np.log10(tier2["n_cm3"])

print(f"TIER 2 (half-Heusler): {tier2['sample_id'].nunique()} samples, {len(tier2)} points")

tier2_out = []
for sgn, grp in tier2.groupby(np.sign(tier2["S_measured"])):
    T = grp["T"].values
    n_m3 = grp["n_measured"].values
    S_abs = grp["S_measured"].abs().values
    mstar = fit_mstar(T, n_m3, S_abs) if len(grp) >= 3 else 1.0
    pred = spb_seebeck(n_m3, T, mstar)
    g = grp.copy()
    g["carrier_type"] = np.where(sgn > 0, "p-type", "n-type")
    g["mstar_over_me"] = mstar
    g["S_baseline_V_per_K"] = np.sign(g["S_measured"].values) * pred
    g["S_residual_V_per_K"] = g["S_measured"] - g["S_baseline_V_per_K"]
    tier2_out.append(g)
tier2 = pd.concat(tier2_out, ignore_index=True)

resid2_uVK = tier2["S_residual_V_per_K"] * 1e6
print(f"TIER 2 SPB baseline fit: RMSE={np.sqrt(np.mean(resid2_uVK**2)):.1f} uV/K")
for ct, g in tier2.groupby("carrier_type"):
    print(f"    {ct}: n={len(g)} pts, m*={g['mstar_over_me'].iloc[0]:.3f} m_e, "
          f"RMSE={np.sqrt(np.mean((g['S_residual_V_per_K']*1e6)**2)):.1f} uV/K")
tier2.to_csv("tier2_halfheusler_baseline_residuals.csv", index=False)


TIER 2 (half-Heusler): 55 samples, 880 points
TIER 2 SPB baseline fit: RMSE=257.6 uV/K
    n-type: n=666 pts, m*=1.886 m_e, RMSE=290.7 uV/K
    p-type: n=214 pts, m*=0.515 m_e, RMSE=99.5 uV/K


## Step 5: independent cross-check against DFT

Same SPB physics, run backwards: instead of "given m*, predict Seebeck," this
solves "given the DFT-computed Seebeck, what m* would that imply." No
fitting to experimental data is involved here at all — this is a check on
whether the experimental fit is in the right physical ballpark.

The DFT database evaluates each compound at a few fixed carrier-concentration
levels, including a very lightly-doped 1e16 cm⁻³ point where this inversion
is poorly conditioned. Both "all points" and "restricted to the degenerate,
comparable 1e20 cm⁻³ level" are shown, so the instability itself is visible
rather than hidden.

In [10]:
def mstar_from_S_T_c(S_uVK, T, c_cm3):
    S_VK = abs(S_uVK) * 1e-6
    n_m3 = c_cm3 * 1e6
    eta = eta_of_S(S_VK)
    F0_val = fermi_dirac(0.0, eta)
    Nc_needed = n_m3 / F0_val
    return (Nc_needed / (4*np.pi))**(2.0/3.0) * H**2 / (2*KB*T) / ME

import openpyxl
wb = openpyxl.load_workbook(MERGED_XLSX_PATH, read_only=True, data_only=True)

for sheet, label in [('ricci_dft_clean_halfheusler_1', 'half-Heusler'), ('ricci_dft_clean_skutterudite_1', 'skutterudite')]:
    ws = wb[sheet]
    rows = list(ws.iter_rows(values_only=True))
    cols = rows[0]
    df = pd.DataFrame(rows[1:], columns=cols)
    print(f'=== {label} (n={len(df)} DFT entries) ===')
    for prefix, ctype in [('Sᵉ.p', 'p-type (hole)'), ('Sᵉ.n', 'n-type (electron)')]:
        v, T, c = f'{prefix}.v [µV/K]', f'{prefix}.T [K]', f'{prefix}.c [cm⁻³]'
        sub = df[[v, T, c]].apply(pd.to_numeric, errors='coerce').dropna()
        mstars_all, mstars_1e20 = [], []
        for _, row in sub.iterrows():
            try:
                m = mstar_from_S_T_c(row[v], row[T], row[c])
                if np.isfinite(m) and 0 < m < 1000:
                    mstars_all.append(m)
                    if np.isclose(row[c], 1e20):
                        mstars_1e20.append(m)
            except Exception:
                pass
        mstars_all, mstars_1e20 = np.array(mstars_all), np.array(mstars_1e20)
        print(f'  {ctype}: all c-levels n={len(mstars_all)} median={np.median(mstars_all):.3f}   '
              f'c=1e20 only n={len(mstars_1e20)} median={np.median(mstars_1e20) if len(mstars_1e20) else float("nan"):.3f}')
    print()


=== half-Heusler (n=153 DFT entries) ===
  p-type (hole): all c-levels n=153 median=0.153   c=1e20 only n=58 median=0.226
  n-type (electron): all c-levels n=153 median=0.428   c=1e20 only n=58 median=0.944

=== skutterudite (n=22 DFT entries) ===
  p-type (hole): all c-levels n=22 median=0.093   c=1e20 only n=5 median=0.361
  n-type (electron): all c-levels n=22 median=0.003   c=1e20 only n=4 median=1.213



## Step 6: does the residual relate to fabrication process?

First the raw text-mined categories (10 groups, several very small),
then a physically-grounded merge into 3 groups by actual synthesis
mechanism (pressure/solid-state vs. melt-based vs. vapor/deposition),
because several of the raw categories have too few samples (as few as 8)
to fit or test reliably on their own.

In [11]:
per_sample = (tier1_final.groupby('sample_id')
                .agg(family=('family', 'first'),
                     fab_cat=('fab_cat', 'first'),
                     resid=('S_residual_V_per_K', lambda x: np.mean(np.abs(x))))
              .reset_index())
per_sample = per_sample.dropna(subset=['fab_cat'])
per_sample['log_resid'] = np.log(per_sample['resid'] + 1e-12)
per_sample['z'] = per_sample.groupby('family')['log_resid'].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() > 0 and len(x) > 1 else 0.0)

counts = per_sample['fab_cat'].value_counts()
keep_cats = counts[counts >= 5].index.tolist()
sub = per_sample[per_sample['fab_cat'].isin(keep_cats)]
print(f"Raw categories with n>=5: {keep_cats}  (total {len(sub)} samples)")
groups = [g['z'].values for _, g in sub.groupby('fab_cat')]
H, p = stats.kruskal(*groups)
print(f"Kruskal-Wallis, 10 raw categories: H={H:.2f}, p={p:.4f}, df={len(groups)-1}")


Raw categories with n>=5: ['PressureSintering', 'Sintering', 'HighPressureSynthesis', 'MeltGrowth', 'MeltQuench', 'Other', 'FilmGrowth', 'VaporGrowth', 'Melting', 'FluxGrowth']  (total 371 samples)
Kruskal-Wallis, 10 raw categories: H=16.61, p=0.0551, df=9


In [12]:
MECH_MAP = {
    'PressureSintering': 'Pressure/solid-state', 'Sintering': 'Pressure/solid-state',
    'HighPressureSynthesis': 'Pressure/solid-state',
    'MeltGrowth': 'Melt-based', 'MeltQuench': 'Melt-based', 'Melting': 'Melt-based', 'FluxGrowth': 'Melt-based',
    'FilmGrowth': 'Vapor/deposition', 'VaporGrowth': 'Vapor/deposition',
}
per_sample['mechanism'] = per_sample['fab_cat'].map(MECH_MAP)
mech = per_sample.dropna(subset=['mechanism'])
print(mech.groupby('mechanism')['z'].agg(['count','mean','std']).sort_values('mean').to_string())
groups = [g['z'].values for _, g in mech.groupby('mechanism')]
H, p = stats.kruskal(*groups)
print(f"\nKruskal-Wallis, 3 mechanistic groups: H={H:.2f}, p={p:.4f}, df={len(groups)-1}")

from itertools import combinations
print("\nPairwise Mann-Whitney (uncorrected):")
for g1, g2 in combinations(mech['mechanism'].unique(), 2):
    a = mech.loc[mech['mechanism']==g1, 'z']; b = mech.loc[mech['mechanism']==g2, 'z']
    u, p2 = stats.mannwhitneyu(a, b)
    print(f"  {g1} (n={len(a)}) vs {g2} (n={len(b)}): p={p2:.4f}")


                      count      mean       std
mechanism                                      
Pressure/solid-state    250 -0.103839  0.938421
Melt-based               76  0.192549  0.994856
Vapor/deposition         28  0.206922  0.992693

Kruskal-Wallis, 3 mechanistic groups: H=5.16, p=0.0757, df=2

Pairwise Mann-Whitney (uncorrected):
  Pressure/solid-state (n=250) vs Vapor/deposition (n=28): p=0.0706
  Pressure/solid-state (n=250) vs Melt-based (n=76): p=0.0921
  Vapor/deposition (n=28) vs Melt-based (n=76): p=0.8460


## Step 7: XGBoost — how predictable is the residual, and does knowing the process help?

Two models, evaluated with cross-validation that's grouped by `sample_id`
(so multiple points from the same sample never end up split across train
and test — that would leak information and make the score look better than
it really is):

- **Model A** (baseline): predict the residual from family, carrier type,
  temperature, and carrier concentration only — no manufacturing info.
- **Model B**: the same, plus the 3-group fabrication mechanism, run on the
  *same* subset of samples that actually have a fabrication label (so the
  comparison is apples-to-apples, not just "more data helps").

If Model B clearly beats Model A on that identical subset, the fabrication
process is carrying real predictive signal beyond what family/carrier-type
already explain. If it barely helps, the earlier borderline p-values were
about as good as it gets.

In [13]:
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, r2_score

work = tier1_final.copy()
work['mechanism'] = work['fab_cat'].map(MECH_MAP)
work['target'] = np.log(np.abs(work['S_residual_V_per_K'] * 1e6) + 1.0)  # log(|residual in uV/K| + 1)

def run_cv(df, feature_cols, cat_cols, n_splits=5):
    X = df[feature_cols].copy()
    for c in cat_cols:
        X[c] = X[c].astype('category')
    y = df['target'].values
    groups = df['sample_id'].values
    gkf = GroupKFold(n_splits=min(n_splits, df['sample_id'].nunique()))
    preds = np.zeros(len(df))
    for train_idx, test_idx in gkf.split(X, y, groups):
        model = xgb.XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                                  enable_categorical=True, tree_method='hist', verbosity=0)
        model.fit(X.iloc[train_idx], y[train_idx])
        preds[test_idx] = model.predict(X.iloc[test_idx])
    rmse = np.sqrt(mean_squared_error(y, preds))
    r2 = r2_score(y, preds)
    return rmse, r2, model  # last-fold model returned just for feature importances

# Same subset for both models: only samples that have a fabrication mechanism
sub = work.dropna(subset=['mechanism']).copy()
print(f"Comparing on {sub['sample_id'].nunique()} samples with a known fabrication mechanism ({len(sub)} points)\n")

rmse_A, r2_A, _ = run_cv(sub, ['family', 'carrier_type', 'T', 'log10_n'], ['family', 'carrier_type'])
print(f"Model A (no process)      : CV RMSE(log scale)={rmse_A:.3f}, R2={r2_A:.3f}")

rmse_B, r2_B, model_B = run_cv(sub, ['family', 'carrier_type', 'T', 'log10_n', 'mechanism'], ['family', 'carrier_type', 'mechanism'])
print(f"Model B (+ mechanism)     : CV RMSE(log scale)={rmse_B:.3f}, R2={r2_B:.3f}")
print(f"\nImprovement from adding process: R2 {r2_A:.3f} -> {r2_B:.3f} "
      f"({'real improvement' if r2_B - r2_A > 0.02 else 'negligible/no improvement'})")

print("\nFeature importances (Model B, last CV fold):")
for name, imp in sorted(zip(['family','carrier_type','T','log10_n','mechanism'], model_B.feature_importances_),
                         key=lambda x: -x[1]):
    print(f"  {name:15s}: {imp:.3f}")


Comparing on 354 samples with a known fabrication mechanism (4815 points)

Model A (no process)      : CV RMSE(log scale)=0.958, R2=0.240
Model B (+ mechanism)     : CV RMSE(log scale)=0.954, R2=0.245

Improvement from adding process: R2 0.240 -> 0.245 (negligible/no improvement)

Feature importances (Model B, last CV fold):
  family         : 0.392
  log10_n        : 0.197
  mechanism      : 0.176
  carrier_type   : 0.152
  T              : 0.083


## Step 8: symbolic regression, restricted to real physical variables only

This is the interpretable counterpart to XGBoost. It is **not** given the
fabrication label at all — only temperature and log carrier concentration,
which are genuine physical quantities a formula can meaningfully use. The
question this answers: can an actual formula, using only real physics
inputs, get anywhere close to what XGBoost achieves once XGBoost is allowed
to see the process label? If yes, the process label wasn't adding much once
you already know the physics. If XGBoost (with the label) clearly wins,
that's the honest evidence that "how it was made" carries information a
physically-restricted formula can't reach.

In [14]:
from gplearn.genetic import SymbolicRegressor

# Restrict the function set to operations that have a physical reading
# (no arbitrary trig, no exponentials that blow up) -- keeps the result a
# formula you could actually put in a thesis, not an unreadable expression.
PHYSICAL_FUNCTIONS = ('add', 'sub', 'mul', 'div', 'sqrt', 'log', 'abs', 'neg')

X_sym = sub[['T', 'log10_n']].values
y_sym = sub['target'].values
groups = sub['sample_id'].values

gkf = GroupKFold(n_splits=min(5, sub['sample_id'].nunique()))
sym_preds = np.zeros(len(sub))
last_model = None
for train_idx, test_idx in gkf.split(X_sym, y_sym, groups):
    est = SymbolicRegressor(population_size=2000, generations=15,
                             function_set=PHYSICAL_FUNCTIONS,
                             parsimony_coefficient=0.001, random_state=0, n_jobs=-1, verbose=0)
    est.fit(X_sym[train_idx], y_sym[train_idx])
    sym_preds[test_idx] = est.predict(X_sym[test_idx])
    last_model = est

rmse_sym = np.sqrt(mean_squared_error(y_sym, sym_preds))
r2_sym = r2_score(y_sym, sym_preds)
print(f"Symbolic regression (T, log10_n only): CV RMSE(log scale)={rmse_sym:.3f}, R2={r2_sym:.3f}")
print(f"\nBest discovered formula (last fold):\n  target = {last_model._program}")

print(f"\n--- Comparison summary ---")
print(f"Model A  (XGBoost, no process)         : R2={r2_A:.3f}")
print(f"Model B  (XGBoost, + process label)    : R2={r2_B:.3f}")
print(f"Symbolic (T, log10_n only, no process) : R2={r2_sym:.3f}")


Symbolic regression (T, log10_n only): CV RMSE(log scale)=1.119, R2=-0.037

Best discovered formula (last fold):
  target = abs(log(0.024))

--- Comparison summary ---
Model A  (XGBoost, no process)         : R2=0.240
Model B  (XGBoost, + process label)    : R2=0.245
Symbolic (T, log10_n only, no process) : R2=-0.037


## Optional: symbolic regression using extracted physical process numbers

For the small subset of samples whose fabrication *comment* (free text)
actually contains a real number — a sintering temperature, pressure, or
duration, e.g. `"SPS(873K, 15min, 50MPa)"` — those numbers are genuine
physical quantities, unlike the category label itself. This subset is
small, so treat this as an exploratory check, not a headline result.

In [15]:
temp_pat = re.compile(r'(\d+(?:\.\d+)?)\s*(K|C)\b')
press_pat = re.compile(r'(\d+(?:\.\d+)?)\s*(MPa|GPa)\b')
time_pat = re.compile(r'(\d+(?:\.\d+)?)\s*(hr|h|min|day)\b', re.IGNORECASE)

def extract_num(pattern, text):
    m = pattern.search(str(text))
    return float(m.group(1)) if m else np.nan

work['proc_temp'] = work['fab_comment'].apply(lambda s: extract_num(temp_pat, s))
work['proc_press'] = work['fab_comment'].apply(lambda s: extract_num(press_pat, s))
work['proc_time'] = work['fab_comment'].apply(lambda s: extract_num(time_pat, s))

has_num = work.dropna(subset=['proc_temp', 'proc_press', 'proc_time'], how='all')
n_samples_with_num = has_num['sample_id'].nunique()
print(f"Samples with at least one extractable process number: {n_samples_with_num} "
      f"({100*n_samples_with_num/work['sample_id'].nunique():.1f}% of Tier 1)")
print("This is small -- treat any result from this subset as exploratory only.")

if n_samples_with_num >= 20:
    extra = has_num.dropna(subset=['proc_temp'])  # temperature is the most complete of the three
    X_extra = extra[['T', 'log10_n', 'proc_temp']].fillna(extra[['T','log10_n','proc_temp']].median()).values
    y_extra = extra['target'].values
    est = SymbolicRegressor(population_size=2000, generations=15, function_set=PHYSICAL_FUNCTIONS,
                             parsimony_coefficient=0.001, random_state=0, n_jobs=-1, verbose=0)
    est.fit(X_extra, y_extra)
    print(f"\nFormula using T, log10_n, AND extracted process temperature ({len(extra)} points):")
    print(f"  target = {est._program}")
else:
    print("Too few samples to fit anything meaningful here -- skipping.")


Samples with at least one extractable process number: 44 (4.6% of Tier 1)
This is small -- treat any result from this subset as exploratory only.

Formula using T, log10_n, AND extracted process temperature (392 points):
  target = sqrt(log(mul(X0, X0)))


## How to read this notebook's results

- **Steps 1-4** should print the same headline numbers as the rest of this
  project: Tier 1 around 967 samples / 37 families / ~12,000 points, RMSE
  around 139 µV/K; Tier 2 (half-Heusler) around 55 samples, RMSE around
  258 µV/K split by carrier type. If your numbers differ noticeably, look
  for a difference in the input files, not the code.
- **Step 5** should show the half-Heusler p-type experimental value sitting
  inside the DFT interquartile range, and the n-type value sitting above
  it — that mismatch is a real, reportable finding, not an error.
- **Step 6** should show a consistent direction (pressure-based methods
  closer to ideal behavior) but a p-value hovering around 0.05-0.1 — a
  trend, not yet a proven result.
- **Step 7 vs Step 8** is the actual decision point: compare Model A, Model
  B, and the symbolic result's R² values printed at the end of Step 8. If
  Model B clearly beats both Model A and the symbolic result, that's
  evidence the process label carries real signal a physics-only formula
  can't capture. If all three land close together, the honest conclusion
  is that family and carrier type were already doing most of the work, and
  process isn't adding much on top — which is itself a useful, reportable
  finding, not a failure.